# Clickbait: PhoBERT level (Kaggle GPU)
Fine-tune `vinai/phobert-base-v2` on ViClickbait-2025 (headline only) and score the VnExpress dev
headlines. Ablation: word segmentation (pyvi) on or off. Three seeds per configuration.

Input: private Kaggle dataset built with `python -m src.classifiers.clickbait kaggle-bundle`
(files `viclickbait.csv`, `dev_headlines.csv`). Settings: GPU T4 x1 or P100, Internet on.

Output (download from /kaggle/working and put in `results/clickbait/kaggle/`):
`dev_scores_phobert_seg.csv`, `dev_scores_phobert_noseg.csv` (mean score over seeds),
`phobert_runs.json` (metrics per seed), and the model of the best configuration in safetensors.

In [ ]:
!pip install -q pyvi

In [ ]:
import glob, json, os, random
import numpy as np, pandas as pd, torch
from pyvi import ViTokenizer
from sklearn.metrics import average_precision_score, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DATA = os.path.dirname(glob.glob("/kaggle/input/**/viclickbait.csv", recursive=True)[0])
OUT = "/kaggle/working"
MODEL = "vinai/phobert-base-v2"
SEEDS = [42, 43, 44]
EPOCHS, LR, BATCH, MAX_LEN = 4, 2e-5, 32, 64
device = "cuda"

train = pd.read_csv(f"{DATA}/viclickbait.csv")
dev = pd.read_csv(f"{DATA}/dev_headlines.csv", dtype={"id": str})
print(len(train), train.y.mean(), len(dev), dev.y.mean())

In [ ]:
def prep(texts, segment):
    texts = [str(t) for t in texts]
    return [ViTokenizer.tokenize(t) for t in texts] if segment else texts

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def encode(tok, texts, labels=None):
    enc = tok(texts, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    items = [{k: enc[k][i] for k in enc} for i in range(len(texts))]
    if labels is not None:
        for it, y in zip(items, labels):
            it["labels"] = torch.tensor(int(y))
    return items

def collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

def run(segment, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
    tr = DataLoader(encode(tok, prep(train.title, segment), train.y), batch_size=BATCH, shuffle=True, collate_fn=collate)
    dv = DataLoader(encode(tok, prep(dev.title, segment)), batch_size=64, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(tr)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    model.train()
    for _ in range(EPOCHS):
        for b in tr:
            b = {k: v.to(device) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward(); opt.step(); sch.step(); opt.zero_grad()
    model.eval(); scores = []
    with torch.no_grad():
        for b in dv:
            b = {k: v.to(device) for k, v in b.items()}
            scores.append(torch.softmax(model(**b).logits, -1)[:, 1].cpu().numpy())
    return model, tok, np.concatenate(scores)

In [ ]:
runs, mean_scores, best = [], {}, (-1, None)
for segment in (True, False):
    name = "phobert_seg" if segment else "phobert_noseg"
    per_seed = []
    for seed in SEEDS:
        model, tok, s = run(segment, seed)
        pred = (s >= 0.5).astype(int)
        m = {"config": name, "seed": seed, "f1": f1_score(dev.y, pred), "pr_auc": average_precision_score(dev.y, s),
             "predicted_positive_share": float(pred.mean())}
        print(m); runs.append(m); per_seed.append(s)
        if m["pr_auc"] > best[0]:
            best = (m["pr_auc"], name)
            model.save_pretrained(f"{OUT}/phobert_clickbait_best", safe_serialization=True)
            tok.save_pretrained(f"{OUT}/phobert_clickbait_best")
        del model; torch.cuda.empty_cache()
    s = np.mean(per_seed, axis=0)
    pd.DataFrame({"id": dev["id"], "score": s.round(4), "pred": (s >= 0.5).astype(int)}).to_csv(
        f"{OUT}/dev_scores_{name}.csv", index=False)

json.dump({"runs": runs, "best_single_run": best[1], "epochs": EPOCHS, "lr": LR, "batch": BATCH,
           "max_len": MAX_LEN, "model": MODEL}, open(f"{OUT}/phobert_runs.json", "w"), indent=2)
pd.DataFrame(runs).groupby("config")[["f1", "pr_auc", "predicted_positive_share"]].agg(["mean", "std"])